# Multimodal Learning & Data Fusion

This notebook covers:
1. **Audio-Visual Fusion**: Combining audio and visual modalities
2. **Audio-Tabular Fusion**: Audio features + structured data
3. **Cross-Modal Attention**: Learning alignments between modalities
4. **Feature Concatenation Strategies**: Best practices for combining features

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# AUDIO-VISUAL FUSION
# =====================================================================

class AudioVisualFusion(nn.Module):
    """
    Early fusion: Combine audio and visual features before processing.
    Simple concatenation + shared encoder.
    """
    
    def __init__(self, audio_dim, visual_dim, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        self.audio_dim = audio_dim
        self.visual_dim = visual_dim
        
        # Feature projection layers
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )
    
    def forward(self, audio_features, visual_features):
        """
        Args:
            audio_features: [batch_size, audio_dim]
            visual_features: [batch_size, visual_dim]
        Returns:
            logits: [batch_size, n_classes]
        """
        # Project both modalities
        audio_proj = self.audio_proj(audio_features)
        visual_proj = self.visual_proj(visual_features)
        
        # Concatenate
        combined = torch.cat([audio_proj, visual_proj], dim=1)
        
        # Classification
        logits = self.fusion(combined)
        return logits


class BimodalAttentionFusion(nn.Module):
    """
    Late fusion with cross-modal attention.
    Each modality learns to attend to the other.
    """
    
    def __init__(self, audio_dim, visual_dim, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        
        # Unimodal encoders
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Cross-modal attention: Audio attends to Visual
        self.audio_to_visual = nn.Linear(hidden_dim, hidden_dim)
        self.visual_to_audio = nn.Linear(hidden_dim, hidden_dim)
        
        # Fusion gates
        self.audio_gate = nn.Linear(hidden_dim * 2, hidden_dim)
        self.visual_gate = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # Classification
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )
    
    def forward(self, audio_features, visual_features):
        """
        Args:
            audio_features: [batch_size, audio_dim]
            visual_features: [batch_size, visual_dim]
        Returns:
            logits: [batch_size, n_classes]
        """
        # Encode each modality
        audio_enc = self.audio_encoder(audio_features)  # [batch, hidden]
        visual_enc = self.visual_encoder(visual_features)  # [batch, hidden]
        
        # Cross-modal attention weights
        audio_query = self.audio_to_visual(audio_enc)  # [batch, hidden]
        visual_key = self.visual_to_audio(visual_enc)  # [batch, hidden]
        
        # Compute attention scores
        audio_attn_score = torch.sum(audio_query * visual_key, dim=1, keepdim=True)  # [batch, 1]
        audio_attn_weight = torch.sigmoid(audio_attn_score)  # [batch, 1]
        
        # Gated fusion
        audio_fused = torch.cat([audio_enc, audio_attn_weight * visual_enc], dim=1)
        audio_out = self.audio_gate(audio_fused)  # [batch, hidden]
        
        visual_fused = torch.cat([visual_enc, (1 - audio_attn_weight) * audio_enc], dim=1)
        visual_out = self.visual_gate(visual_fused)  # [batch, hidden]
        
        # Final fusion
        combined = torch.cat([audio_out, visual_out], dim=1)  # [batch, hidden*2]
        logits = self.classifier(combined)
        
        return logits


In [ ]:
# =====================================================================
# AUDIO-TABULAR FUSION
# =====================================================================

class AudioTabularFusion(nn.Module):
    """
    Fuses audio embeddings with tabular features.
    Common in: voice biometrics + metadata, speaker identification + demographics
    """
    
    def __init__(self, audio_dim, tabular_dim, hidden_dim, n_classes, 
                 audio_depth=2, dropout=0.5):
        super().__init__()
        
        # Audio feature processing (deeper)
        audio_layers = []
        prev_dim = audio_dim
        for _ in range(audio_depth):
            audio_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        self.audio_encoder = nn.Sequential(*audio_layers)
        
        # Tabular feature processing (simpler)
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tabular_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion with attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            batch_first=True,
            dropout=dropout
        )
        
        # Classification
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )
    
    def forward(self, audio_features, tabular_features):
        """
        Args:
            audio_features: [batch_size, audio_dim]
            tabular_features: [batch_size, tabular_dim]
        Returns:
            logits: [batch_size, n_classes]
        """
        # Encode each modality
        audio_enc = self.audio_encoder(audio_features)  # [batch, hidden]
        tabular_enc = self.tabular_encoder(tabular_features)  # [batch, hidden]
        
        # Stack for multihead attention
        combined = torch.stack([audio_enc, tabular_enc], dim=1)  # [batch, 2, hidden]
        
        # Cross-attention
        attn_output, _ = self.attention(combined, combined, combined)
        
        # Extract and concatenate
        audio_attn = attn_output[:, 0, :]  # [batch, hidden]
        tabular_attn = attn_output[:, 1, :]  # [batch, hidden]
        fused = torch.cat([audio_attn, tabular_attn], dim=1)  # [batch, hidden*2]
        
        logits = self.classifier(fused)
        return logits


In [ ]:
# =====================================================================
# FEATURE FUSION STRATEGIES
# =====================================================================

class MultimodalFusionStrategies:
    """
    Common fusion strategies for multimodal learning.
    """
    
    @staticmethod
    def early_fusion(features_list):
        """
        Early fusion: concatenate raw features directly.
        Pros: Fast, captures interactions early
        Cons: Different modalities have different scales/ranges
        """
        return torch.cat(features_list, dim=1)
    
    @staticmethod
    def late_fusion(outputs_list, weights=None):
        """
        Late fusion: average predictions from each modality.
        Pros: Modality-specific models, interpretable
        Cons: Slower, misses early interactions
        """
        if weights is None:
            weights = [1.0 / len(outputs_list)] * len(outputs_list)
        
        fused = torch.zeros_like(outputs_list[0])
        for output, weight in zip(outputs_list, weights):
            fused += weight * output
        return fused
    
    @staticmethod
    def gated_fusion(features_list, gate_net):
        """
        Gated fusion: learn importance weights for each modality.
        gate_net: outputs importance weights [0, 1]
        """
        concatenated = torch.cat(features_list, dim=1)
        gates = gate_net(concatenated)  # [batch, n_modalities]
        
        # Normalize gates
        gates = F.softmax(gates, dim=1)
        
        # Weighted sum
        fused = torch.zeros_like(features_list[0])
        start_idx = 0
        for i, features in enumerate(features_list):
            end_idx = start_idx + features.size(1)
            fused += gates[:, i].unsqueeze(1) * features
            start_idx = end_idx
        
        return fused
    
    @staticmethod
    def tensor_fusion(features_list):
        """
        Tensor Fusion Network (TFN): outer product of all modalities.
        Captures higher-order interactions.
        For 2 modalities: fusion = outer_product(f1, f2).flatten()
        """
        if len(features_list) == 2:
            f1, f2 = features_list
            # Outer product: [batch, dim1, dim2]
            fused = torch.einsum('bi,bj->bij', f1, f2)
            # Flatten: [batch, dim1*dim2]
            fused = fused.reshape(fused.size(0), -1)
            return fused
        else:
            raise NotImplementedError("TFN for >2 modalities requires careful implementation")
    
    @staticmethod
    def bilinear_fusion(f1, f2, bilinear_layer):
        """
        Bilinear fusion: f1^T * W * f2 + b
        Captures pairwise interactions between modalities.
        """
        # Bilinear: [batch, 1]
        fused = torch.bmm(f1.unsqueeze(1), bilinear_layer(f2).unsqueeze(2)).squeeze()
        return fused


In [ ]:
# =====================================================================
# DEMONSTRATION
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test 1: Audio-Visual Fusion
    print("=" * 60)
    print("AUDIO-VISUAL FUSION")
    print("=" * 60)
    
    audio_dim = 128
    visual_dim = 256
    hidden_dim = 128
    n_classes = 10
    batch_size = 16
    
    model_av = AudioVisualFusion(audio_dim, visual_dim, hidden_dim, n_classes).to(device)
    
    audio = torch.randn(batch_size, audio_dim).to(device)
    visual = torch.randn(batch_size, visual_dim).to(device)
    
    model_av.eval()
    with torch.no_grad():
        logits = model_av(audio, visual)
    
    print(f"Audio shape: {audio.shape}")
    print(f"Visual shape: {visual.shape}")
    print(f"Output shape: {logits.shape}")
    print(f"✓ Audio-Visual Fusion working!\n")
    
    # Test 2: Bimodal Attention Fusion
    print("=" * 60)
    print("BIMODAL ATTENTION FUSION")
    print("=" * 60)
    
    model_attn = BimodalAttentionFusion(audio_dim, visual_dim, hidden_dim, n_classes).to(device)
    
    model_attn.eval()
    with torch.no_grad():
        logits = model_attn(audio, visual)
    
    print(f"Output shape: {logits.shape}")
    print(f"✓ Bimodal Attention Fusion working!\n")
    
    # Test 3: Audio-Tabular Fusion
    print("=" * 60)
    print("AUDIO-TABULAR FUSION")
    print("=" * 60)
    
    tabular_dim = 50
    model_at = AudioTabularFusion(audio_dim, tabular_dim, hidden_dim, n_classes).to(device)
    
    tabular = torch.randn(batch_size, tabular_dim).to(device)
    
    model_at.eval()
    with torch.no_grad():
        logits = model_at(audio, tabular)
    
    print(f"Tabular shape: {tabular.shape}")
    print(f"Output shape: {logits.shape}")
    print(f"✓ Audio-Tabular Fusion working!\n")
    
    # Test 4: Fusion Strategies
    print("=" * 60)
    print("FUSION STRATEGIES COMPARISON")
    print("=" * 60)
    
    f1 = torch.randn(batch_size, 128)
    f2 = torch.randn(batch_size, 64)
    
    strategies = MultimodalFusionStrategies()
    
    early = strategies.early_fusion([f1, f2])
    late = strategies.late_fusion([f1, f2])
    
    print(f"Early fusion shape: {early.shape}")
    print(f"Late fusion shape: {late.shape}")
    print(f"✓ Fusion strategies working!\n")
    
    print("✅ All multimodal fusion models working correctly!")
